# ETL Notebook
---

**Goal**
* this notebook is used to test the pipeline extraction and transformation
  * API extraction -> `raw_geomaterials.parquet`
  * bronze layer -> `bronze_geomaterials.csv`
  * silver layer
    * geomaterials -> `silver_geomaterials.csv`
    * context
      * minerals_hand -> `silver_minerals_hand.csv`
      * minerals_crystal -> `silver_minerals_crystal.csv`
      * minerals_optical -> `silver_minerals_optical.csv`
      * rocks -> `silver_rocks.csv`

In [1]:
import pandas as pd
import os
import duckdb

from scripts.dirs import DATA_DIR
# pd.set_option("display.max_columns", None)

#### raw_geomaterials

In [6]:
df_raw_geomaterials = duckdb.read_parquet(os.path.join(DATA_DIR, 'raw_geomaterials.parquet'))

query_rg = "select *from df_raw_geomaterials"
duckdb.query(query_rg).show(max_width=10000)

┌───────┬─────────────┬──────────────────────────────────────┬─────────────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────┬───────────┬───────────┬───────┬────────────┬─────────┬───────────┬────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────┬─────

In [8]:
query_rg_counts = """
  select 
    count(distinct id) as qtIds,
    count(*) as qtLines,
    count(distinct guid) as qt_guids,
    count(distinct name) as qtNames 
  from df_raw_geomaterials
"""

duckdb.query(query_rg_counts).show()

┌───────┬─────────┬──────────┬─────────┐
│ qtIds │ qtLines │ qt_guids │ qtNames │
│ int64 │  int64  │  int64   │  int64  │
├───────┼─────────┼──────────┼─────────┤
│ 55193 │   55193 │    54559 │   55192 │
└───────┴─────────┴──────────┴─────────┘



In [19]:
df_raw_geomaterials.to_df()['id'].nunique()

55193

In [20]:
df_raw_geomaterials.to_df()['guid'].value_counts(sort='desc')

guid
                                        635
464e5cfa-be77-4568-a724-62137f35df18      1
d65f8cd8-2d4f-449f-982f-d0b8ccde7d05      1
15cf6052-f36d-4c79-9549-178f8752f4cd      1
3fb7fc00-3fe4-4b75-a7e8-527fb60d04ed      1
                                       ... 
e0e3940a-491b-4f9a-9c48-d449c0ab3ebd      1
327c8877-cd5c-48c7-ad8b-644b99848fa0      1
579f8340-b46a-456f-b4fe-bfea2b04b5ba      1
d00d508a-1521-42ca-838d-a34e24b114ae      1
1d830e22-6a2d-49fa-b7f3-d21ddeb4d179      1
Name: count, Length: 54559, dtype: int64

raw layer:
* 55193 entries on raw_geomaterials
* 55193 distinct `id`
* 54559 distinct `guid`
  * 635 blank
* 55192 distinct `name`
  * '毒重石' name has two entries

#### bronze_geomaterials 

In [25]:
df_bronze_geomaterials = duckdb.read_csv(os.path.join(DATA_DIR, 'bronze_geomaterials.csv'), sample_size=-1)

In [27]:
type(df_bronze_geomaterials)

duckdb.duckdb.DuckDBPyRelation

In [23]:
df_bronze_geomaterials.show(max_width=10000)

┌───────┬─────────────┬──────────────────────────────────────┬─────────────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────┬───────────┬───────────┬───────┬────────────┬─────────┬───────────┬────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────┬─────

In [33]:
query_bg_counts = """
  select 
    count(distinct id) as qtIds,
    count(*) as qtLines,
    count(distinct guid) as qt_guids,
    count(distinct name) as qtNames 
  from df_bronze_geomaterials
"""

duckdb.query(query_bg_counts).to_df()

,qtIds,qtLines,qt_guids,qtNames
0,55193,55193,54558,55192


In [36]:
df_bronze_geomaterials.to_df()['guid'].value_counts(sort='desc')

guid
464e5cfa-be77-4568-a724-62137f35df18    1
0fa1e006-f09a-4a00-80f7-e64253466897    1
15cf6052-f36d-4c79-9549-178f8752f4cd    1
3fb7fc00-3fe4-4b75-a7e8-527fb60d04ed    1
f56fe7a8-7df9-4a76-8d05-9787c1776e65    1
                                       ..
e0e3940a-491b-4f9a-9c48-d449c0ab3ebd    1
327c8877-cd5c-48c7-ad8b-644b99848fa0    1
579f8340-b46a-456f-b4fe-bfea2b04b5ba    1
d00d508a-1521-42ca-838d-a34e24b114ae    1
1d830e22-6a2d-49fa-b7f3-d21ddeb4d179    1
Name: count, Length: 54558, dtype: int64

silver layer:
* no blanks `guid`

#### silver_rocks

In [ ]:
df_rocks = duckdb.read_csv(os.path.join(DATA_DIR, 'silver_rocks.csv'))
df_rocks